In [1]:
# --- Imports ---
import numpy as np                          # numeric arrays; used later for np.exp() to convert log-odds coefficients into odds ratios
import pandas as pd                         # DataFrames — how we load, inspect, and reshape the tabular patient data
import matplotlib.pyplot as plt             # plotting (used elsewhere for EDA visuals)

from sklearn.linear_model import LogisticRegression         # our baseline classification model
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report  # evaluation metrics
from sklearn.preprocessing import StandardScaler            # feature scaling, needed specifically for logistic regression


In [2]:
# Load the SQL-cleaned dataset (already had cross-field consistency checks + engineered category columns applied in Stage 1)
df = pd.read_csv("heart_disease_risk_updated.csv")
df.columns


Index(['patient_id', 'age', 'sex', 'resting_bp_systolic',
       'resting_bp_diastolic', 'cholesterol_total', 'hdl', 'ldl',
       'triglycerides', 'fasting_blood_sugar', 'hba1c', 'bmi',
       'resting_heart_rate', 'max_heart_rate_achieved', 'chest_pain_type',
       'exercise_induced_angina', 'st_depression', 'family_history',
       'smoker_status', 'alcohol_units_per_week', 'exercise_minutes_per_week',
       'sleep_hours', 'stress_score', 'wearable_owner', 'daily_steps',
       'diet_quality_score', 'has_heart_disease', 'chol_inconsistency_flag',
       'bmi_category', 'age_bracket', 'bp_category'],
      dtype='object')

In [3]:
# Drop columns that shouldn't go into the model:
# - bmi_category, age_bracket, bp_category: these are just bucketed/binned versions of bmi, age, and blood pressure,
#   which we ALSO keep in raw numeric form. Including both would feed the model the same information twice
#   (a form of multicollinearity), and the raw numeric version lets the model find its own optimal split points
#   instead of relying on our hand-picked bucket boundaries.
# - patient_id: an arbitrary identifier, not a real feature — has no predictive meaning.
# - chol_inconsistency_flag: data-quality metadata from Stage 1 cleaning, not a real clinical predictor.
df = df.drop(["bmi_category", "age_bracket", "bp_category", "patient_id", "chol_inconsistency_flag"], axis = 1)


In [4]:
df.head(10)


,age,sex,resting_bp_systolic,resting_bp_diastolic,cholesterol_total,hdl,ldl,triglycerides,fasting_blood_sugar,hba1c,...,family_history,smoker_status,alcohol_units_per_week,exercise_minutes_per_week,sleep_hours,stress_score,wearable_owner,daily_steps,diet_quality_score,has_heart_disease
0,44,Male,117,74,193,57,106,119,112,5.2,...,False,Never,2.9,86,5.4,19.8,True,7731,62.9,0
1,57,Male,139,94,185,69,110,35,114,5.9,...,False,Never,3.0,132,4.3,45.8,True,2629,74.6,1
2,29,Male,128,78,197,52,108,157,95,5.5,...,False,Current,3.5,128,5.1,17.7,True,9290,65.7,0
3,72,Male,132,86,197,59,104,143,92,4.8,...,False,Never,2.7,18,6.8,63.6,True,7373,48.5,1
4,62,Female,116,75,154,65,75,104,135,6.2,...,True,Former,3.3,24,8.2,58.7,False,6331,47.3,1
5,50,Female,122,71,147,46,101,41,134,5.9,...,True,Former,3.3,190,5.9,55.1,False,6857,82.6,0
6,50,Female,133,85,227,63,124,194,109,5.5,...,False,Never,5.1,91,8.2,82.2,True,8466,63.7,0
7,58,Female,133,81,141,54,99,40,118,5.8,...,True,Current,2.5,92,6.4,45.5,True,6769,42.4,1
8,51,Female,121,64,219,84,104,85,136,5.7,...,False,Former,2.8,266,9.0,65.6,True,7648,53.5,0
9,51,Male,126,77,172,50,104,115,132,6.3,...,True,Former,6.1,89,9.1,58.7,False,7531,53.2,0


In [5]:
# Binary encoding of the 'sex' column: Male = 1, Female = 0.
# We use an explicit .map() here rather than .astype(bool)/.astype(int), because .astype(bool) on TEXT data
# only checks whether the string is empty or not (every non-empty string is "truthy" in Python) —
# it does NOT read what the string actually says. That means BOTH 'Male' and 'Female' would convert to True,
# collapsing the whole column to 1s. Explicitly mapping each text value to the number we want avoids that trap.
df['sex'] = df['sex'].map({'Male': 1, 'Female': 0})


In [6]:
# These three columns were already stored as actual booleans (True/False), not text like 'sex' was —
# so a plain .astype(int) safely converts True -> 1 and False -> 0 with no ambiguity.
# (Unlike the .astype(bool)-on-text trap above, this works correctly because the values are already real booleans,
# not strings that merely look like categories.)
df = df.astype({'family_history': int, 'exercise_induced_angina': int, 'wearable_owner': int})


# Logistic Regression Model

In [7]:
# One-hot encode the remaining multi-class TEXT categories: chest_pain_type, smoker_status.
# Why one-hot instead of simple integer labels (0, 1, 2, 3)? Because these categories have no inherent order/ranking —
# labeling Never=0, Former=1, Current=2 would falsely imply "Current" is mathematically greater than "Former",
# suggesting an order or magnitude relationship that doesn't really exist between unordered categories.
# One-hot encoding instead creates a separate 0/1 column per category (e.g. smoker_status_Current,
# smoker_status_Former), so the model treats each as a genuinely distinct, unordered group.
df = pd.get_dummies(df, columns = ["chest_pain_type", "smoker_status"], dtype = int)


In [8]:
from sklearn.model_selection import train_test_split

# Split features (X) from the target we're trying to predict (y)
X = df.drop(columns=['has_heart_disease'])
y = df['has_heart_disease']

# Train/test split: the model only ever LEARNS from the training set. The test set is held back completely
# and used only ONCE at the end to check how well the model generalizes to patients it's never seen —
# this is the core idea behind everything in ML: we care about performance on unseen data, not memorization.
#
# test_size=0.2 -> 80% train / 20% test
# stratify=y    -> our target is imbalanced (~70% no-disease / 30% disease). Stratifying forces BOTH the train
#                  set and test set to preserve that same ~70/30 ratio, instead of leaving it to random chance.
# random_state=42 -> fixes the "random" split so re-running this cell always produces the same split (reproducibility)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)


In [9]:
# Scaling data - puts every feature on the same footing (mean 0, standard deviation 1) so the model judges
# each feature by its actual relationship to the target, not by the coincidence of its measurement scale
# (e.g. cholesterol ~100-300 vs sleep_hours ~4-9). This matters specifically for logistic regression, which
# fits a weighted sum of features — tree-based models like random forest don't need this step.
#
# IMPORTANT (avoiding data leakage): we .fit_transform() ONLY on the training data (learns mean/std from train),
# then only .transform() the test data using those SAME learned values. If we let the scaler see the test set
# while computing its statistics, information about "unseen" data would leak into preprocessing — even though
# we haven't touched the model yet. The test set must stay genuinely untouched until final evaluation.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + apply on train
X_test_scaled = scaler.transform(X_test)         # apply the SAME fitted scaler to test — no fitting here


In [10]:
# Sanity check: after scaling, every feature's mean should be ~0. Values here appear as things like 1e-16 or 1e-17
# rather than a clean 0 purely due to floating-point rounding limits in how computers store decimals —
# that's expected and confirms scaling worked correctly (a real problem would look like 0.15 or 3.2, not 1e-16).
X_train_scaled.mean(axis = 0)


array([-8.43769499e-17, -2.44249065e-17,  3.06421555e-16,  3.37507799e-16,
       -2.78665979e-16,  1.80411242e-16,  7.54951657e-17, -1.37667655e-16,
        1.93178806e-16, -6.37268016e-16, -2.22044605e-16,  5.28466160e-16,
       -1.50990331e-16, -2.66453526e-17, -2.22044605e-16,  0.00000000e+00,
       -4.88498131e-17,  1.24344979e-16,  1.07913678e-15,  3.61932706e-16,
       -2.66453526e-17,  6.66133815e-17,  3.28626015e-16, -6.21724894e-17,
       -4.44089210e-18,  2.22044605e-17, -6.43929354e-17,  5.10702591e-17,
        1.77635684e-17, -1.33226763e-17])

In [11]:
# Logistic regression, conceptually: it computes a weighted sum of the features, then converts that sum into
# LOG-ODDS of the outcome (mathematically, log-odds is the scale on which a plain weighted sum makes sense).
# Exponentiating that internally converts it into a probability between 0 and 1 (via the sigmoid function),
# which is then thresholded (default 0.5) into an actual yes/no prediction.
logistic_model = LogisticRegression()
logistic_model.fit(X_train_scaled, y_train)     # learn from TRAINING data only

y_pred = logistic_model.predict(X_test_scaled)  # predict on TEST data (never seen during training)


In [12]:
# Overall accuracy. Important context: our target is ~70% no-disease / 30% disease, so a model that just
# guessed "no disease" for every patient would already score ~70% accuracy without learning anything real.
# Anything meaningfully above that baseline suggests the model is actually learning something.
print(accuracy_score(y_test, y_pred))


0.915


# Precision and Recall
Recall — of all patients who actually have heart disease, what fraction did the model correctly catch? This is the number you care about most here, since it directly measures how many real cases you're missing (false negatives).

Precision — of all patients the model flagged as having heart disease, what fraction actually do? This measures how often a positive flag is a false alarm.

In [13]:
# confusion_matrix for binary classification is read as:
# [[TN  FP]
#  [FN  TP]]
# classification_report gives precision/recall/F1 per class already computed from those same counts.
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[135   6]
 [ 11  48]]
              precision    recall  f1-score   support

           0       0.92      0.96      0.94       141
           1       0.89      0.81      0.85        59

    accuracy                           0.92       200
   macro avg       0.91      0.89      0.90       200
weighted avg       0.91      0.92      0.91       200



[135   6]

 [11  48]

           [   precision    recall  f1-score   support

           0       0.92      0.96      0.94       141
           1       0.89      0.81      0.85        59
  
    accuracy                           0.92       200
  
    macro avg          0.91      0.89      0.90       200
    weighted avg       0.91      0.92      0.91       200 ]


135 True Negatives, 11 False Negatives, 6 False Positives, 48 True Positive

11 patients who actually have heart disease were flagged as negative by the model

48/59 patients were correctly flagged as having heart disease


# Random Forests Model

In [14]:
from sklearn.ensemble import RandomForestClassifier
# Random forest, conceptually: instead of one model fitting one set of weights (like logistic regression),
# it builds MANY individual decision trees, each trained on a random subset of data/features, then lets them
# vote (majority prediction wins). Individual trees tend to overfit and make different mistakes from each other;
# averaging many differently-wrong trees cancels out a lot of that individual noise.


In [15]:
# Random forest splits on raw thresholds rather than a weighted sum of scaled values, so — unlike logistic
# regression — it doesn't need scaled features. We pass the original unscaled X_train/X_test directly.
rf_model =  RandomForestClassifier(random_state = 42)
rf_model.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [16]:
y_pred_rf = rf_model.predict(X_test)


In [17]:
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


[[134   7]
 [ 14  45]]
              precision    recall  f1-score   support

           0       0.91      0.95      0.93       141
           1       0.87      0.76      0.81        59

    accuracy                           0.90       200
   macro avg       0.89      0.86      0.87       200
weighted avg       0.89      0.90      0.89       200



In [18]:
# Model overfitted: compare accuracy on data the model TRAINED on vs. data it's never seen (test set).
# Train accuracy of 1.0 (perfect) alongside noticeably lower test accuracy is the classic signature of
# overfitting — the model memorized quirks of individual training patients rather than learning
# generalizable patterns, so it doesn't hold up on new data.
train_acc = accuracy_score(y_train, rf_model.predict(X_train))
test_acc = accuracy_score(y_test, y_pred_rf)
print("Train accuracy", train_acc)
print("Test accuracy", test_acc)


Train accuracy 1.0
Test accuracy 0.895


# Overfitting Fix

Unconstrained random forest will keep splitting trees deeper and deeper until it separates every training example. The fix is to constrain it by using k-fold cross-validation on the training set. This method will split the data into k chunks, trains on k-1 of them and validates on the set that is not being trained, then rotates which chunk is held out, and averages the results.

In [19]:
from sklearn.model_selection import GridSearchCV


In [20]:
# Constrain the forest so it can't keep growing deep enough to memorize the training set:
# max_depth caps how deep each tree can grow; min_samples_leaf requires a minimum number of patients
# per leaf before a further split is allowed. Both force trees to stop at general patterns instead of
# chasing individual data points.
param_grid = {
    'max_depth': [3,5,7,10,None],
    'min_samples_leaf': [1,3,5,10]
}

# GridSearchCV tries every combination of the params above using k-fold cross-validation, evaluated
# ONLY on the training set (never touches the test set) — this avoids "leaking" test-set feedback into
# our hyperparameter choices, which would make the test set no longer truly "unseen."
#
# cv=5           -> 5-fold cross-validation: splits training data into 5 chunks, trains on 4, validates on the
#                   held-out 1, rotates which chunk is held out, and averages results across all 5 rounds.
# scoring='recall' -> optimize for catching real disease cases (minimizing false negatives), rather than
#                     plain accuracy, since missing a real case is clinically worse than a false alarm.
grid_search = GridSearchCV(
    RandomForestClassifier(random_state = 42),
    param_grid,
    cv = 5,
    scoring = 'recall'
)
grid_search.fit(X_train, y_train)   # fit only on training data — test set stays untouched until final check

print(grid_search.best_params_)


{'max_depth': 10, 'min_samples_leaf': 5}


In [21]:
# grid_search automatically refits a final model using the best-found params on the full training set
# (best_estimator_) — no need to manually rebuild a RandomForestClassifier with these values ourselves.
final_model = grid_search.best_estimator_
y_pred_final = final_model.predict(X_test)     # the ONE legitimate, final look at the test set

print(confusion_matrix(y_test, y_pred_final))
print(classification_report(y_test, y_pred_final))

# Re-check the train/test accuracy gap to confirm tuning actually fixed the overfitting (rather than
# just hiding it) — a much smaller gap here than the untuned version means the fix genuinely worked.
train_acc_final = accuracy_score(y_train, final_model.predict(X_train))
test_acc_final = accuracy_score(y_test, y_pred_final)

print(train_acc_final)
print(test_acc_final)


[[136   5]
 [ 15  44]]
              precision    recall  f1-score   support

           0       0.90      0.96      0.93       141
           1       0.90      0.75      0.81        59

    accuracy                           0.90       200
   macro avg       0.90      0.86      0.87       200
weighted avg       0.90      0.90      0.90       200

0.95125
0.9


*Logistic Regression is the better model

# Seeing which factors contribute to heart disease the most

In [26]:
# Logistic regression's coefficients tell us each feature's effect on the LOG-ODDS of heart disease,
# holding every other feature constant (unlike Cohen's d/correlation earlier, which only looked at one
# variable at a time in isolation).
#
# feature_names comes from X_train (pre-scaling) since scaling turned the data into a plain NumPy array,
# losing the column names — scaling doesn't reorder columns, so pairing them back up this way is safe.
feature_names = X_train.columns
coefficients = logistic_model.coef_[0]        # one coefficient per feature, same order as feature_names
odds_ratio = np.exp(coefficients)             # converting log-odds coefficients into odds ratios:
                                               # exp(coefficient) tells us the multiplicative change in ODDS
                                               # for a one-standard-deviation increase in that feature
                                               # (recall: features were scaled, so "one unit" = one std. dev)

results = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'odds_ratio': odds_ratio
})

# Rank by ABSOLUTE coefficient value, not the raw signed value — a coefficient of -1.2 (strongly protective)
# and +1.2 (strongly risk-increasing) represent equally strong effects, just pointed in opposite directions.
# Sorting by the raw signed value would incorrectly push all protective (negative) features to the bottom.
results['abs_coefficient'] = abs(coefficients)
results = results.sort_values(by = 'abs_coefficient', ascending = False)

print(results)


                             feature  coefficient  odds_ratio  abs_coefficient
12           max_heart_rate_achieved    -2.402297    0.090510         2.402297
13           exercise_induced_angina     0.992334    2.697524         0.992334
14                     st_depression     0.744504    2.105397         0.744504
0                                age    -0.725743    0.483965         0.725743
15                    family_history     0.429504    1.536496         0.429504
1                                sex     0.392518    1.480704         0.392518
27             smoker_status_Current     0.359300    1.432327         0.359300
5                                hdl    -0.341356    0.710806         0.341356
4                  cholesterol_total     0.323127    1.381441         0.323127
10                               bmi     0.314579    1.369683         0.314579
29               smoker_status_Never    -0.242090    0.784985         0.242090
19                      stress_score     0.238117   

# Conclusion

  The strongest independent predictor was reduced cardiac fitness — patients with lower max_heart_rate_achieved showed substantially elevated risk (odds ratio 0.09 per standard-deviation increase, the largest effect in the model). Exercise-induced angina (OR 2.70) and ST depression (OR 2.11) were the next-strongest independent factors, followed by: family history (OR 1.54), sex (OR 1.48), current smoking (OR 1.43), total cholesterol (OR 1.38), and BMI (OR 1.37).

  Notably, two variables that appeared prominently in the earlier single-variable analysis — age and LDL cholesterol — showed much weaker independent effects once other correlated variables were accounted for. This is attributable to multicollinearity: age correlates strongly with max heart rate (-0.74), and LDL correlates strongly with total cholesterol (0.85), so much of their individual importance was actually shared with, and reassigned to, these correlated variables once all were modeled together.

  **Overall conclusion:** heart disease risk in this dataset is driven primarily by cardiovascular fitness markers and direct cardiac symptoms, with family history, sex, smoking, cholesterol, and BMI contributing secondary, independent effects, and the relationships involved are close to linear rather than requiring complex non-linear modeling to capture.